# roboflow에서 데이터 내려받기

In [11]:
from roboflow import Roboflow
from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv())

private_key = os.getenv("ROBOFLOW_API_KEY")

In [13]:
# !pip install roboflow

rf = Roboflow(api_key=private_key)
project = rf.workspace("allys-workspace-knezs").project("mnist_numbers-m28t1")
version = project.version(1)
dataset = version.download("yolov8")             

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Mnist_numbers-1 in yolov8:: 100%|██████████| 44/44 [00:00<00:00, 2828.69it/s]


In [3]:
import os

dataset_path = "Mnist_numbers-1"  #dataset 압축해제한 폴더이름

for split in ['train', 'valid', 'test']:
    images = os.listdir(f"{dataset_path}/{split}/images")
    labels = os.listdir(f"{dataset_path}/{split}/labels")
    print(f"{split}: 이미지 {len(images)}개, 라벨 {len(labels)}개")

#data.yaml 확인
with open(f"{dataset_path}/data.yaml", 'r') as f:
    print(f.read())

train: 이미지 12개, 라벨 12개
valid: 이미지 2개, 라벨 2개
test: 이미지 2개, 라벨 2개
names:
- '0'
- '1'
- '2'
- '3'
- '4'
- '5'
- '6'
- '7'
- '8'
- '9'
nc: 10
roboflow:
  license: MIT
  project: mnist_numbers-m28t1
  url: https://universe.roboflow.com/allys-workspace-knezs/mnist_numbers-m28t1/dataset/1
  version: 1
  workspace: allys-workspace-knezs
test: ../test/images
train: ../train/images
val: ../valid/images



# YOLOv8 nano 모델로 학습

In [5]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")  #nano 모델 (가장 빠름)

results = model.train(
  data="Mnist_numbers-1/data.yaml",
  epochs=50,
  imgsz=640,
  batch=16,
  project="runs_numbers",
  name="mnist_exp1"
)

New https://pypi.org/project/ultralytics/8.4.32 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.31  Python-3.11.15 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=Mnist_numbers-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, 

In [6]:
from ultralytics import YOLO

# 학습된 모델 로드
model = YOLO("runs/detect/runs_numbers/mnist_exp1/weights/best.pt")

# 동일 이미지로 추론
filename = '000948.jpg'
results = model.predict(filename, conf=0.5)

for result in results:
  for box in result.boxes:
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    conf = box.conf[0].item()
    cls = result.names[int(box.cls[0])]
    print(f"{cls} {conf:.4f}")


image 1/1 c:\Users\Admin\hipython\object\000948.jpg: 640x640 (no detections), 24.0ms
Speed: 4.6ms preprocess, 24.0ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)
